# Brazil FDI Observatory — Python robustness laboratory
**Prepared extension; not a completed empirical sensitivity study. Fernando Barbosa.**
Use a standard free CPU. Do not connect a GPU or purchase compute. This is a new notebook;
it does not run the Financial Intelligence models, LangSmith experiments or the baseline 2,000-draw bootstrap.
The baseline study and all prior archives remain read-only.
Read Research & Methods before interpreting responses as effects.

## Questions and assumptions, before computation
1. Are remaining residual patterns seasonal or serially dependent? Test squared-residual quarter indicators (seasonal volatility) and auxiliary residual-vector lag dependence; report the entire equation family with Holm adjustment. Quarter-mean tests would be tautological after fitting quarter indicators.
2. Do LP responses depend on additional lags? Compare 0, 1 and 2 additional lags with HAC bands. These reuse generated baseline shocks, omit some uncertainty and do not implement a new identification design.
3. How do units and growth accumulation change interpretation? Normalize each saved draw to a +1 percentage-point FDI/GDP impact; accumulate GDP growth and convert log differences to level percentages draw by draw.
4. How does a finite-order model behave when the true system contains moving-average dynamics? Repeated synthetic DGPs, 119/240 observations, VARMA coefficients 0/0.5, fitted lags 1/2, 1,000 deterministic replications per cell. Report bias, coverage, Monte Carlo standard error and failures.
These are prespecified for this new extension, after the original baseline was observed. They are not retroactive preregistration.
No municipal treatment registry currently supports an empirical ATT. HonestDiD-style sensitivity, a municipal Monte Carlo battery, alternative inflation deflators and new omitted-variable specifications remain separate unfinished work—not claimed implemented here.

## Books and papers actually informing the checks
Pearl & Mackenzie (2018), *The Book of Why*, Chapters 1 and 4: association versus intervention; common causes, mediators and colliders.
Freedman (2009), *Statistical Models: Theory and Practice*, Chapters 1, 6 and 8: stability under interventions, economic units and bootstrap assumptions.
Verbeek (2017), *A Guide to Modern Econometrics*, Sections 2.5.7, 5.2.4 and 9.2: power/p-values, reverse causality and Granger prediction.
Matheus Facure's Python handbook: graphical causal models, doubly robust estimation and staggered DiD.
LP interpretation/inference: Plagborg-Møller & Wolf (2021), Montiel Olea & Plagborg-Møller (2021), Li et al. (2024).
Municipal design: Callaway & Sant'Anna (2021), Sant'Anna & Zhao (2020), Sun & Abraham (2021), Roth (2022), Rambachan & Roth (2023).
The workbench bibliography includes DOI links and the two multinational-investment applications. Private books are not embedded or redistributed.

In [ ]:
from pathlib import Path
import base64,hashlib,io,json,sys,subprocess,zipfile,shutil
PROJECT=Path('/content/brazil-fdi-robustness-source')
raw=base64.b64decode('UEsDBBQAAAAIAAAAISigieWcVxQAAF80AAAZAAAAc2NyaXB0cy9mZGlfcm9idXN0bmVzcy5wea1ba2/cxpL9bsD/gXE+NGlxKI1seWNOuIgdO/cGEO46tm8W2MGAaJE9M7T4MpuUZqTVf99T/SA5D8W+izsIIrLZXV1dXY9T1e1nz569SdOszaqS586vH/45qcp86zTVVSfbUkjpyJavhAyct1yKPCuFs8xyIR3eCKcRPFX9g6dP/lE5PPnaZVLR8p2sXIpGlIlwiioVufSdS16uPhVZu3YSnufSqRrnytK8qqpWtg2vHbERSUckQPPZs2dPnyybqnBq3q7z7MrJirpqWucDXp8+MS+8WdW8kcJ31lxSL9/5IokFuQah3Hfuspp49s1o9AgK0fKUt7wnUnZFvXW4dMq6b6t5maIF/9Vp3whxtFIvKeB1Rl9lYbgcf1PPQdHlbdYK2VrOVUOdC2qTZtgyzQI9yvZKhcxWWMEya2O5la0ofAhb1lUpBSSZV5BgXDfVF5GQqNBU5zwrnz55+uTPNx9/f/P28v2naM7KZZIxn63SOl411W27ppdlshzeGiGaOFljZwTewEiMzmzx9Mmn9+/fRedn56+mZy+mT5+kYglxcrf2QjDSdk1phR2g+fzilUtbgs8B6UR8tcX6XM8L1mKTZlCf1vUMEX4j3Nq/4XknvPDpEwe/OjKDZ3WAnRRlGxTXada4+kVGn5tO+GKTyTaurtWbp0c2/DaivQ5SbJ90lRBcTduXEGR8LbZmeFamoBWd+9C96jYueRn9xnMpvABKCuG7hiRkXUd1cAs9jWW3XGYblwVtUTNvRp+C2wb7aRaI6U1rIzB3gpUZKlxKgX1UvOUVT8HbrmSiaMyslk1W1l0Lql1pJYNHLRtq003LqE41qUTeUPspqzE7JJWefu1404oG9ohvTC15EydVHp2ZwUU04kgP1op3WnLtAgLqwMw2tmLTuko8WbmKWNcuf2Le7gqLOSNNF2wRRSypCqXazIHl0KferNlizrDULKV+52dnZztEcuysuwwUv5CMesUqa9FkVRo3pJ328/xs4dvHyRTPjfgasT/AMZftthYupttjMReluwTZ6fS14mugBJanr1+/+uOc7XwAXXyB8l/88ZLtkCrrgDcN38YCos7Bk5F40Fax8iCu54PxD4rx34mY5XvEqBljmNxGy3lvs4uB0BKb1Jo+dxEmxj52RRlD2sm1O1/O56yT8Y5pL0W67MpUWoNeHJAj5laijWEuRQY97Bfgp01Vx8uska22r/2RC29fEJlcZiVswd1C9nnuekqG4w935oMe+aPzUSTwVW3TJS15syyFDKWjXOCnN3++f+cklYDBJRkZve+UFXy7U4rbIUzAHwaa2ibSPtLd+nc+NC3nK7bwZl20ndu3cDHZ/EI7JtWeuWgfT8DskuS6Sq4liRhz8HwVyCqHk9odCcfMkzYueNtkGwz1u+CzF3w2Zqo94tIvfGKn8zVJbdWw85JMzQfDrU+OIkuUscloCkM4auoz6qtf6ckwmi2VTBBiIUWO0OqOqcHcW4+C6rjx52nY8EwK50/yM++bpmpc9qGiIH0jxj0h+66kbUEIhythu9tNTMAlaMF4P0TgcXjV+17t94IexI3IOc0Ut5W7M8Y6M0jseRyNHJ9up9giZHQvQwo65Kek5yyxNkTI0pk/6vGOe7PFg6YK7Vtmq+ie3YhG4jsLpz6TAqsNKdz5zOoZCzUDPqPIEOsIxxQvcUxQIo49n6WiFhRUkm1M3Vh4XwclL0SoA6Xit1b8qo3sR5oYd0rh9tSghXrL/L/oJduqAQqjbosHn0EVrwmU0ZThIagJzPp2mWDKnBmNJliDB5lkqmEEWxT5ddVkd1AeFipPrP3v9IXn+VqO9sfyOkaEaHgMe0Pv+Zk/9c8XIMCT+Aqz3GYpHFPI1s7JYMM1XtQoR43ap2kdA5yYgA1h9N/f/OpI2miRTuxXx2z8BM6VNBirJC3k3SbLM95se/8yuQFEwlfM5fw3z1MnzfiqrGSbJXLm/L3KC4dCPQQkeLKGg7kRE3LuyiaWvMjy7QGLMgOKUz2wAWxsbnFNcErkOQt3TJOh5xzxxz9/ifjFCk7CCvzgAi+E8Yz8SHoPB7NhdzqSg1X6mfKJGpZbg4VEabqsHfAzNjYBxCho6CdRKpPP2q2j5ardKxHKCBdlS8Nr4ADI7/jcPoYHzNiRHtFuoz0IOEJi2tT2MNgAtfaAofJ/mIZ8entqyc/D6avFjNq/Fw4qcEkDgIngdbX1+1CQ1jLUs24f/OoKHu5GAGZUtzLSWMG6J0OLYPGQEJERejswcM9wd3DeToAgcjoo7Gn5GPPtR5Gxd5zVkY1us6/RAD8yZATzOhzFezOx6RDtxf2v34z3Nmb/oQk4heClTvp4TgvcUixuRUrpwISSGSBEp11XCDW6623VwdquhNPyrq3yagUFy/vgfYBn0FBBui5twFd4GsM4QroUHNsYzRd4ajKe40lTgZdaUcAeIwHkEGk0PbkjlazFfLqYXUfb/sUIGO7gi0+emvyiwKpFAwTr9ijM7oTqnLWRLIL/uvzkdvPQ/7J4/vzc33gB2qFTNzFhzohcFPPp9fo2RdiCgW+0Sb/0gdIE4kPT6FSNhSTuB2+Y4TYiVHML3xSTHAl3iK1wX3rzabjwJaTGG63yRGlpspZhvJYPclGKRi6CG4eUrnLEI1qi9u4Z+TsWqq11b4O+iaLYkoUvEBSGr7XOScY8YuZoFAtSzztx5+kJ1vj8WsmT3CuEqT/X3oKwypcoOnMQU4QzHuqnJ/Xz697E7IZcqhj10v9pEe76P1BeifRQYbr55QTfQlKYzqPHxSEnU//yZNpjV/trxOqQntWm+WVIgl74euL9sYTBeqCo0SAlKNeAYisCRmLVa9sR5PURPSewfw1Bey8wCkpsbz5EtEH/LkkBfZrpe/Xv8tv6pyUCD5+pDiSZO9FUyA+vn1/64wV53mzUEdYwQY9wERmNxcse2dsI7I9UezT6OxRbKzeZ/OPKTa6QFvkdSk6rOaLmPiPHz6weXVrRkC7hi1/fkDrdZbWrDc3fqeC48wZolGnVa6in7rXwAcbWVRoBTOUF80h6IcjN1XuMEZHmo74ZzXfUCDQIiebNeA6SCuliY6JBFF0udg1ql3dN5Fu8617/Ou8mvN2bUkA4FAIIX5M8EOnKjmARU5lzD99kLRLCHipoWJQ3Sg5dO9y5qQhy5YRgZNIIUXo2QwQsErx1VJWtWpq4ZGfQ0egQwikJWp6QpRo/c4AbxxmjQ3bhWKg4YE2LF6lG13LkvgiFWSv7AEUOB3QBho4xoteH7EKrl2GN3ukvQBzXcE4DYS63Rd1WUOsRA/vAVkFaZyVKFdhG8pw5sqDFSk77o5MPaFOufYOTYA/SjJKN1pHZnSAwCAljwRskv3rsRI91dFnSSB1y4p2kXaIajwaShBYJ6iBJkAP8/P8gnapr+6BPuqqzBzIVnXCMjAUDovpEdZhBGZs2QsvweadqgC++6uPNRC0jzcNctUyApUajjuCVjY8xQCeUpwOaIPcGihyBE8vrehSMXnh7sa1UeBOEvMn6MMz8HG02ve89uTgSTn4vVYVyP5iQtiE1IFvAQ1oV+0EFxkxMJ9kOu5blLwPLF/scqz4DJNpqcZ2sQ/P3pKQItdnMw3Lx3UFqTUDi+wKV4d4GBEJOyAp4IeeTixDbkWTjTwT6Y2yNy/N6zaPg7MLT/XaJUtnCjAJtr99M25hkowFQxiEcjbNf9eyzrEGsoUoPqysQusWexa8vYqTELLSEH77HbxrPqDNIFmLefd9xVXXI5JstJXiEZvuUbbB8rdQzKn30hzzkRqqCHJT6OiHwXuiEtysT0ZALa7eBcwkkxbtVAeXSX3kOiO6kFXSLPC91zZZboH7hIPPEbF2ZQYGKyXD2g29VIwrtSKgQqksm7X7WaZ0FZXBpnDb8NpbIVRDtxUFytFswmiUd0jep9LnTefmN2HEX2aDP56OCG/2UnyQ8V/HU7atvp0um0jK1gUMxadJnwqf3WXh28RCU9R0z5wl1llznwmAYOhRK9iwHehElc6UecBeqmhjhBS7spf9ycWj91W6RVY9Q9T39+HM0FZPp2VGvoLwwSDRw2NmdrlTs6byWmtVkYiT0QzByaif6sT9wctoKUfZk6tS189u730//9u6D4WGfpBW/JUvAcFMXU3rAV+yomWmKmbzTKTbjOf3PHAORuKO9siuJ63uspdcbA+X0+oDuSnFr289GBsSgq22sDACU+rM1pwZkoKWKJiG9XwnN1rDwsqxulERnvYCFBgQ0i3O11aH/tjQyYgeTAi1A9XNVvoM6Id0NsWjbajj3v/p8k8noTJfwvqpwF5ydX/jB9JUf/PTSD17/x8VOtUid8+XiRuSxYZ+FR3ZAredgDx6hM3Jhj3FrN/0vOD7gdOS5fu0JDAqngol0eGtk6LhrkA2cd5DwhNiB6EvydJwAWmnKnTPMuAZowsjeE+66mVOk+KN6lnJKqpjX8lJUHY0rUzk4o77OF4+qeW7pw6u13K+pmG+9yY/O5zWC7cQmKA4SHbgcjG8cfeTRNfC97/72YYZPndC+18mkg/Cf5QQeyeQNeCU+bTw3NZOmVEkr/BgCegD2ONiO0eoOrZ8Q7T8BjZLrdeeqpl3SsYCrGFamprle2PT7TWSPtNz5PHh14Z/5Zwt/HkzP/OAldu7sgt7OSOvO/eDFxcIa49vxwGlgx6HTT4YChmDYq36EiMBsoG3GJXQZueXJxdnUf4G8ctunnXGeXQtXjLKidpzPk21vPS/czttF9OYX/JkAHL39ReD9RC9TvUxstWcbbeeYJVzM7kapbemfYVaFf6PhkN2UkPrTFvV9zhqeZp1ki/+MpmHvhJY8y7uG3FBHpzG05cPm9ZXSZunHUe9dXEXQn557s7z2qaayf5Cvj7NML39U2oR+kdb0JgJw16valDz0SMm0ap2T/ayjl0Z/8DRDl3YduQcFjLq6FY37xl97v7w1Qny8z2SKXp6qVZuNBQTO6/laYfGZTCMpzMue31bTs1D98ZmBHcKgJYkHmdKJC8yZjmRgp7mLT5Np8PrVc5n+HKmBP0doOzFtBycSkEk8ENZpKoUczQ/CwcjzfLDOzTjyCQGjS3LxGqXtgCbAH0t3QFU9zpqoltMhkxoBKW3XSDS3Dja65isa0PuYosJsccKbvFLV5scPCH903lOUGZ/ZwXskaANHI2iVA7r9ppVTXd7ZkvkQRsNHwL5SHZaACpI861qEhC+RO6CpVO7bHlmMyxFkkaQiOgFThxh7UGc4c9rLzuxPldr3Uw87EqwNBj+WxrFchH4IFivRRrpWXyQM8K28Lx8mBb9XjD5M6vv6gZrvQU4BN3UwcJwazF4TDNQJg3Qfm5Z+A0wcX6ww43fOAWaUcw3d50wLnR0Rgv2Zs9fxIHMSuYj+4uhF3yuhROb7jl989nF8DKzOI7rCKTKk+22yfkRMVNf9C8G0zZZKRtF3hdDHyYhNImqE/wHe+oNjuszKN/lKtSrELTZqypFrJrwiNt7D4xOoEyO9X/693RMi41tZh/8+SR9LZulH5jBOQA97KUC/UwKkMaSr/WL11QB8QB5ELhTQleIdlb9Uoc/61VG5T1FdeERGZwyqWP8PJHlHTFb7iCH1RYJe6pNMvZVUWq3pQhngB2Jpun8Cqm//KGyuHqF3hnWD2GlF3mT0/biw2Mgrx3alLLRPYCmRveuHAOTXRhUh1Nfn7nRin73TYS7vmBAeY+Aq43I0QS/hPu4sJnjTwe5Q2sdmOqYYo7NMFSMmKkZMEEJULdGcbZpt+Z4MaVvSRlGFXNXcGR1TS9olRWEnMn5SRUJdKhN0iFcma+Ti1xqjggWgD/xnijWOlaha69uG32V5oOFwj1HUebNTIP9JsppqVHRNwfmoopCz7KMVHXCDcwphn9eIbv3MQ9UhoxVRTULFNIqrI7I0PM9FPmkbaCkRHM69Zdul2z7uXvHkuquVgPsbOHQCrYsA1GpuzIniCulFdE/390aXWagHElXAIpltECN2r/cNIVAdXzervLpy2XOmNp+yenVYbK7OQEfVPcMfIhbcZTUzjXTAgSbN6AQSHqolau+NVyt4mS0Jgt2X+67qZsf76LDuqyMBs6ogA+gFu4YUb5I1FU+01g0n3hPNg2JO91R1E3OtNvifrP6NVmOG++yW+f233z/E797/dvnm8/t3yknf7UGJ4wyFd/qAnRw4eozMY/SB0QwTu35jECP3bL/sO2fv22s4YNXE4ruApqYDFY/A166rNH2kaN07tXvq2JOuelKT5cb73/tjjD/siyVVYrF9rFzMHHsbfaeUzy13w00UpQfXIh7VJkS/8TUnKwe4BdJoFpoGdarmevgTUwoHb6s3Trtw84JRSZPQjRViN1XXjHlKU5uzFt3P3PJyTK+9oqiZ0KBkCrTWTnkd2SvfwZtm1ZEz+EBvjevNkNbzNI25aXfZZNJ0WJ4qQZNx+/aKm1aGYwOqrq279vExlo2DgepODvOTdZUlVJJkWUll3FZJwBzB4DmvbcVqoitT/vhK0cI3yX3UD398xnGMNQwDKPYUVP4w4xFXF6uloLFy58aNup3IA7qfyAO9cDzswO7ZlYBKCnsbT3X+t9zHqxuqTbBPxuuP/9GBvZ4TIsSRB+2vMqwEmnignvxl3sn1eE/gXc23Hwbpjcx4nCq4pucJM3eeh27fzAH0nazdy9S7NCe6iyF99FLQvt9QA8YYn6St+YBF/aoTPUX+myj98Pr5sZzk6NXtXUJmi/5U/gLT226pTTztlhj6tE+j5R3JEjTeiO5HFhHmvLhKebh/KUppmqfsxXbZO0y0PcbGZLsePUroB4yu8Jn+B5n4nhk8zM3mLty9DRwnEXp5xzr8pYLsud9+1w+AnPWSZoeNaT5um7rDg8/6f75TdEQzZTs2+JH+JQ95aOfGbrUOFdjaMVTaNTnYSRzTlscxVCiOyU3HMQu1u6Z/jfJ/UEsDBBQAAAAIAAAAISgYFrVzjxEAAFctAAAVAAAAc2NyaXB0cy9mZGkvbW9kZWxzLnB5nVpbb9w4sn7Pr9BmsZBks9t2MNmdUUdBnOzkZIDFTm5nX4SGIEtUN9O6RZQ67hj+7+crkpKodjuzc4LALZHFqmJVsW7U06dPP3LZtSLteObgoU+7vk0K5z/XH52kyhyR8aoTueDZQm7rdOcUdYrppq2/8LQTdSWXT5583nKn5WnfSrHnjiibJO2cMgHa2wmBdBQC6dRVcXD6KuOt02FhxtMiaUG9NYxopO8TKUW1cTKRbKpadiKVTlYDS1V3DiCTm0LIrcIArrutU+d4qSV3eFpXdSnSI4RPnz59krd16cRx3mOTPI6J1brtsFEgTRTcEzP0RdYVc74lbQUmpF7YJN22EDfDqvd4HcCrvmwOTgLummGogfgwgP9NptfLVDSHJVjv5IAj3YpnZpKGyzrjhVx2MlFgXV0XI2iS5X1R8JY5u0aOfNrLkkYQOVlqjEBRt8mGDwhanmSx3pdM9tw8ZmIDMTEnTaq6EtDtkycZz52mSETl3frBEwf/RO4IaAPEqpR7t6xqlthc2yYHP2g5hFkNC5ZdDb10nu+fXOjRJOv6puD+uDTSa/e+k9ets3dE5dyuTy7PoM1x2R206+38YLZ6x/T6peh4KT3//jH+N7ziMI4H/NO6x7jPizrpfHUuyAqBRchcVFhCkjKI/l1XXK0277daoBmXYlN5B/adNQwY2y4kSCNgPdAQRfUIugqRA71yPWTjBOW0LvqyijGV7rwIA4CWXsFBwl+oBT77HqmHYM3OooN+XgAiLpJNYCDN61oJz7yQANuk2nDvijXnV/567es95KKL5UGShB7ZR8V2IUx8mzR89d/u6TZ8KBt/hb8b3oWHYQurLLw6/65RR1faPG54l4TY+nfe1tKD8oZZtvP9VVr3VSfDSMPS/r5MO9sZhulfL3mojFZPZb7vnDtelJ1DFmc7LRlbKo2/pk19CcNLvRd7McvOmzOQ90f0NyxmmNyxmHgtRJUUm2UhO/nVu40CBuprpneLty9r1sJ/ZVqoIw6QIxR/CUltWAFrSwQo/ycpev5r29at534EwAJqEqmAy3WUUzBm51rcQGYRMIBSeGNkBM/R8CrzBuS+sTUpslBztrh9RQsBvw/V+PLzK/V7QWvUk15E1FhRbzLobtqt1CMelo8niyBfhJcnNoKtL5paio6iicLdI+RgbdIKOolmN9dEwNi/2hUUdrYLsnOvgNGe7ViwXn7W2pvpTq9Oa4SpCi7fMiAojinlrcbZKCA8BLPVpK7HDTQvr4IJbhewYLFTkPzAgWtBPBhJJpnoZajchweAMrmln+RGepOMuNjsk0J6I0rfHxWh/Z1Lu3QD+svcaze4Zi6E4oKJPXMHOUk3UI/MVQfHDdQPcxs3aJibuQFmND8AVL/MvRGpG2jutKLOZV962jb8M2Kx3niTon1L6T4wJ21S8o63wGitA5n6G4Ym2HvtRfDSINLCWykTZdu6Fd+hh6tnrG6RFdj+ZKcMXsFFarNrf6WBrDMHhZE+1PjcyaghhajhbTnoZudHamK9eh3SOGx5UkO6rQsudwePZl7ZlF9pWF/9aqRbYVmP2cf5Fdsp/4PZ6HI9ER3d0NZ2sOMqyyEZj8OuCZAjt+BtghhjmLl218yGNua4fRmSXyeq2/V5eP1KPS3IudtWhNFXr9lrrQqVzMVWMqd88AO1bJM0fAvTmkcrw442s/UqZq/DY9VeGqegEj/bH9TFnnuvmUEx2S5OrL/8/DAsGMAGW9fxQYPchg8i4S3T1GAoYyhCRFghyfmBrhABhsm4EDvuAfykwk6pq1JGqsn6i+1R6DrfBub3vFqvWr4Jb+H2q/W4PK15bpy2fBAhAG+Cgx0XYFfjcnAKhYc2lmixI99nRw+ocG4xP4iHI4joUBWEslz+/q9P3ixE8Y2/xDQ59Lg7NDx0312/cRm97r5lMrxz4eNgejj/2/NJeff+AyKSR1uKRZrY8gbvxL1tspSfSq4tdisoq6U8dbILw3oTToRWJ010ZhDX4XSeViqTMO/Ky1K+Yd6zHxs3b/6sZTeCp1xa1qhzMWOKN4nkQHZskIdpJtIGtYaFmafRUrtJo9rva9SWckcknXJNbRxdIU/s1msV4KMgWx+7oeN0cG4oFj44nQjwiyvgGkcnBzT809ufww9jc2gL8vUZ5Ix5vd+ITgEzVpKJPOctR2JwMsKOGeRiZGoYMPjN65KCV3IrZPhsDL04ORP6l1d88Y8T+cq70SapmEX4VrlLXTl5IgoUtl1N1XFdpXhzj0I6hlGj3vTK+yJEzzhCZDYsI+ofMc/ohMUDXqGq15gTQwjxI8sm5H7bokZBfTCYbhn+bNTYhw8tddWH/aJfljyptDwu/RUl9r1J7EW1tyweb16PENpfVP7qQ3i5PB3nypnlpGEfbclHveqjYLFdX1SrD+eEtGsTlFopJoD3VUp//QuvWmy13D6chdVZpdWeh7uz3ZlXLqZj7597O8q7ppG5tGFdblCq1KgT1FMY0p4PSFeyHJLLkc/sSbfDDFXoS5l7H1iWU7aDTGdbI41yk+xLL6lrUvYFMlXKTTvuUKVdJlXHk35lNyAWI3giD2XT1SAOa9m0nEtqXeR4yOrSNQqzuh5eTumVCcqKzA0SlCkSE3aMdge70qmwhKQ/wk+iRx5KCCMCWaNcj1Xvwps88zfRbcfOxzJNunQbD6/e0ekfwaQoUdPnokAe6LkI23ULW18l4dC28CTTISH8iSV9V9OT+/q3Ny7FaepoAKAleUhJSbmbuqyiABK6BG0VMPaWh8Llzh026ga0Lwbt5PFckQmcPvRHpI5mdnrmBtnQjkO1N3VdKPCXy6tLVe0TxIvl5XPfBDB1nsKjU3V0sm0+VRY+vjF3XOkG6hEW2SADa5GMDbn5cCz161rZrNrf0cSLK8rApSTGH0wp5hWFaDDr9csQGxl7GEl1QBU67l33AW7JdGyOSTrI6rOkxVbcX4cG29DbSxUkLBsetFGeDGIoEyBBJaB7dXw16+QlLXcqcnUSKJ10y5E6McUQklEcB/xPk14mBSl5OBSVZqiIdb8LEa7PDjCapjAcyFlLpe5bhAQFtGz76sJtAEqdxouvPVwob4vDMpV7F/6s6alRoFeMnSe9Mk3AXea5R8RdpleNbZ+xu+ZZFDXsxbB2SfPGlNUhDJtsqRaCD0+TB96M38ZIakOTYJBBS7MPSD4Xm2g0d/LWB+tAS+tE626VQvH9YZ6s10RuL+NN1sQbFGvd1mVuznFgq0zG6ZY8t7t+gJGBaWSCcdaXpeDGP0WuESpMNWvrJs5Fi4z7c9tz/xjBeoytOjt58fOpLsBb/k11iJPK+flSl+stbdkxhORqNAjKEEWpjXDLi8yI2OSASAXmwlN56dAGwEEQGTw39H+6vaX7io0+Eg/RaHNBbObUQg9LUXkTTrbjB1Rl5U2WOGVQRqrSRmXShMOCKUj91fkI/9k5dJCKwkn2SB7UfmvkxO1eG7jiJaU2Nyq5lQP/aSQjYFnUyMrovcSzTMgh62CszDA83p6/endkVaa4IaMSbX4i5X2nhVY0qiT8UfX4jinVr+jQh49FM2WxGiXl9eHJ5F7Ll5Suc+a0LylRA4NnZ8/YkJ/kfI9KRUNFV8H6wnq2EjsohDeZKKW2TOOxv/acQnNouxLVcpu966sHu4Npyw5Bo0O0T5rYXmV0q5yI7YqUx2tqgWJt9A6LEYW7UguW5S4TrcfBeRfXO4vlHDGXt02L9aFu4Htj/x7BULkmijlEz4L1jM8aukGa9wFQv9FoxqexjMfbRG7vTUJcUqGkuLtwtfuHg1ZubTjTBLFUPCNbUHFmco0050euxZK7/ktovZ7wA68HqWgpOgNVpxSypORkOO3DnYaiwu5mVALrhXplRuPUKzOPCLCcBuZaVWND2YoAKjJkWSvK63syx4gKeuqSPkyiFycqQDujHlM1MWXJIzN2ptwYcefunQgun98vq+a7O7BwPKcrDkshRilpM6rkKIGjZE91+JLMSxufLpDSQO10SK68NHJx2uDY8VBQlj2v4qmEEVXPbXqGvYnoILIB6WQTZkY1XY8QtdVG1agworpcIgVIkGrHGPWm0U/Q0CeSG13RRKe0x4TNsSqRpG5iR0CF+qXjMA3pXbKpVWk9+lM9iiS3LxLyFERjljQfFd/Nw8J7huBh9T1rPkTB1HuYF8+P1eQ25nOrszEU2Nb8w4p8vlhLCHDNBNW1h6P2ke4NWTFlRKJiy3F3UsNPuenL8OrEWe8rnSw6oyt0NHZ3jvAmpvgUW/FJ40eAWt3ECE/xieBkMziCH/N5fK+nKPlLRGQ4MyqsHswXjZk+sZ+KDJHgpg0d7QR7wwFvlnQOY9nnubj13GVXNuqYo+5uluTYvscU5qk4Qi5KaxjYCrUYiiZUXKxofEnRR5XQGJgfYw2tQK3jcJvypnO8iWs2lfb/EtV1sVGjyjHwIxsI71wr2LmBINeakO8J6G6W+/erySubc85yf3XsDPLZRZcnYNB/eyy63iiXvw7Dy6OjNRIy0QmqV9Wkdobs7hGvT3ItuBoFXeYqmelbC/WIYKmdqh4bWKfhR+LLUZNTR97cff37758/ff54/d65A6H7i7uRifvAuZvo3es4w/TYSO8eYfSLShldlhe93FrpAISm1kwSyYQc20rWNdk+ulxPl+xqDbLRookfhb96CD/pCp6zpTurUF3Ff/UDrETaBS9ecM9Gyb4OiZrC9pWwRcvLZ8/Z8urvbPnzT2z5yz+er+9H1GDpx9iPeP5TBCh7Cyz096s5uXuT0lfqCvKos2JumY6LgYQaH5TJwQoVCHzcpI7BT81c1pAo68Jtqeo+79afKnB9SUWXEyMnU8ODFqEW/51gAsc9d5njLr8go/TUMup0ACMg5kU6KjyK5cHAxv2UiJiwcqrIsa6mjssIAPirk/vLyRv/gHckLVh874zVuKOIjYxTs25qe+TzlsepTfzVeUMZxKave4ljxxf0CQ6nRoUuhlAw8j2UB7HjwGvzFWQtpqBUySqJEF6+SrIvcKNV56hCTtdRwKlrba2sF+6zy2eXH67G1BfzqtbwX/798jGRRQBaI+arn+b/KTr3vbU7hiK5rfvN1nl2efXLh58mCept/zkhDv4wnMqi8L+rdJT4Jj/2Mlz+8vxsxKKQ132HAgRhQ8JFl0kMbUgVOa50T5a2NjDgqrzVvOiKSztn3epGGYWN6SYRGCkQd3jMC7ERaqeqhzeuJs6oCI2GVhn1+6YeiuoZSob0Q3R4i9x/v33zm6N07OgiDaT+55/vnaLeOLpTwpy/OV8vvtL42zdvT058/PXXj2pCd1LMxMpBrc5DKLTlqRjO5G/Vt6TNnLf//M25sA4EaGKZ6v5pVQZ3w7W+ZYdw6szlpHF7EFkfc+3egQlgBILdTy2TWN/TD/UKvazvme6VN+o31t0KpanojorHUt/F6i8Hhr7GcONfjliUXynJq0w9kTXTlx51oUT9v5+oPiycuXghkjdKauoGPM+J+p47OYdrA7DqUDl0KQ7AD1pccNiUeiT0sUdGVlm3qm+qP0eM9eeIbvCayE9XpeO+deI9fUcRW9+bDDDqMwBzbCiXoqcpcLhTTIHUjtNPUp1aiOzU3SYp9eGooZrpSxu6KE1SZh0uC68dnmad2E+qzT9+omku2nUjlayAnAb5MH1MHL4nWGzonvp8ezIY/AU7Y98FXJ6d0StzG3KOmZwb1fxCNViPn/sNpkZ50uEBGOhZPSC6oEo21l5JNn+YnVmjJ3K0MTkaK01lZpkwXVuY+dByi5VlU/Z0T0X/6GSRtE4v7DhuHjUGjmbX8wa5PlvzL2pH7+5YfFG37/GPYJ23+gLR7pzDbexJpYlR6gIrUBVTx135QOqWW65Wf9aoX3wrG/+jJjUzS4bQNihiaioJuT3RFY/+CPHaNM6RkFvbiocrjLmXNqqeBxpL7yZmqYROE+6rsXF6gjv3bV/Ql2Skw57EPdWZdGQma3u0XHhgevezax8ttSf/B1BLAwQUAAAACAAAACEoE98eYvwHAACNFwAAFgAAAHNjcmlwdHMvZmRpL3N0b3JhZ2UucHnNWF1v47YSfc+vUP1Csaso7eLeRaFAD4ttFl2gLYrdxT7cNBBoiYpZyxJLUk5s1//9zpDUl620SdGHJkAiicOZ4ZmZwyEXi8W7pja8NpdbrkQpeBFow+55kLN8xQNWF0HRKraseBSIuuCSw5/aVLvAyeNIwFS+Eluu48VicVGqZhNkWdmaVvEsC8RGNsqAproxzIim1hf+04rpVSWWUfCbbuooaHQUyIqZslGbKNCr1ogK/u/g817IUlTc6S6Y4UZseKe5e48C/Ltvai8nmUH1ndgv8HpxUfAyqJuHkCaB4uBh3U+P8XOnIW5NTmOhG3SGmZDaiYW459qEMIMlwXJnuB7U+MXEesVe//eNlaHxij/6OU5BzuqmFjmrwi2rWj7MRgTiot1I7UYiDS5na77T6WcFr7zWiCbTuRDpe1ZpHrGqah6ymtX+XXPJFDON0mlIIvhNCKUxr/Om4CFpTXn5HXFeaLblGVoMEaLEIhMF3qOLAH7wewzqINLxZl0IFbqXzptHoU3WrO0btTPMRqZ21oMwq0y3ZSkerfrYPb8iMYgQeg1/4wclDM8sgOEpJPS60bHikAg5D0E4QiXOb8VZcer3CYRVwwrt7FppZ4O6+ZhCGcZpMt+6v0pP4udWhYtxYDSQ+CFRS0IDpoPSzcIfSJAgX7X1GsojgGWpsGKbZcGS0noQfvvN6/98jX9otCSEJqu4lZhyoZ3kzHQpNMmXi7xiWgefTFvsnDlcQ5aJWpgsCzWvyihQTWO6AErV/MZzM1kX/qBkjIIpDoT4RME33VRbHtJrO+znOgn/MhKa6sqbuhT36RCOsYorArM4MsJVWYgrje7HKEUGLYioTm+lBU8icCcqdK6ENBo1EBrfV80yJF/HcgfoC6xfE8i4ZlCzwFXKaAxTSCDSJiP07lUF2Rk+S+NHKJFXt3/ivuK/t0LxDeZ+bB4NiZ5Um0kheSWAO8DPu1PECpd5qQ/vkPUHCTAD6wF/ZqaZeE1jpjPZaKgkmoySlw64IU3wwuazpkd6Eqgl0zwTyNfC7GZMExdJkozCGhF0tv/k/R6rrpocaSbt8wpggCgLrslVSQ7nlo8u/lPfereGNPKK6S3pBskdxtt/jy3tQDUHHPhuZoUnGd/WYxfhFfybmL5Nvn1zNynksqkKrhDYW6LYA5CoVEirvIDHDYBRaXiAjTFfy0ZATsAbf8TtBZ8APSBpfIK0NK0md0nYuXLlVNNn0in++EwfVPSw+HrqAUkGRn9KODoMoCYTFFzAM8d7p3GP5lNE7swKlCawOcfQB2jY1eGj37tJ0j3F3UMIudlTGJRXlemmVTlQM6odcZXieaMKfcotdkEQkCs/7ZRQdM2kXgHDHdbJgXRL2cb33ITdK+3Dkmxv+whFpFWV/YL/7442DdbRFnPAOxMDq28A5eMQmT8pKS15Dn1RblsdD9skS8ENtwgY9G6PqwsIDbw+z9z5pIUccRO+SjugBjAHdS/Iuq4UHL14nbHyjEnoVL33QUKrlCFDhTPjLu7apM6XqxnCAxP0GmVe1nTMeIIVYxUNtWEbSUhdubMeygjH5xUAf4/bBCv4VTqm3kQxAeTzBXuVG6UaFZIbtCTqe5vY2D/b8EKzWJZQGmRqqacmB8b1lAvPmKyLxr+KmF7INs8kGl8V/fhZbXXVQuk/zUpzi3r+vja72JFm391NYuva0fqeK6kgXL6bE7VsAXvoV5PucPKcdgGr59mtgrPR9wruVAKJU3gf7Olv4srQtqVDvH0iWWBwxuke37dpfR36FdnDSi9WjJhenkFWeAIHW5zQNIWwb2TF4cUdTN2wcxXHrXtjWP1irTScl8IhYdd0oCw7PIA2kknT7XhD8Aab1liL0eFIu82BjsMqvJbIgenlox5TYCDDRKXTn+GoOe7UZwrrCaAh6RwsyQBKRJghiT3dRk/v9h6u5Cmwon59yd9PNq/iGBG/VKhn9xCAyOE4agfamm1hAC8SxqBBXuCtwD+K1MjUFCxnjCTe6F87vWT5upXOX9DHK7upiNru+qfOLhaLT567gqaudj5FuEs6HQdf8Cpl112j2HTkUJPBhm+WwPpLDrhy347YjabNgSnddUtnxPcN9gA38sUemEbvo97ZUZxbiSYnil6wHdhFTE5znmpmugd60i3YtY7Oc7wu/GnOXRcMnc6G1aLELgKCC37NJybtbw7Ir79CiK/IUwlqnR46Og9916KUBA50l4fz08Lx8mDDfYz3Qo4w43jHxNQu9Yom1yAEhd16+hn2ZsFfa8X/E/I9wtGriQhs8f3oh1+y72/e//j28833ERY7bOW64ltepW/sbcQ+me/e7BqTvbtugb7nZcBN+xavBVWQtx/f/fDhy83lT29//vD+5tNnX2fDJtXFaqRjfKnTr9KDNUjZO8Vd5r+HZ+Ow8raCFCB44AUKA286mahv+4eAD2P2HogkXXSQEEIK/zIt9hw6KFtoIFDxevA+It2daDbiilzlQCWSaW2br9xdn1qDaKIbOc5w1XxydEb8oSZyizzbDN1nd4N5gpMcLnxm88qOn2TKkhXpPsaIgnQ4OXXC0HmbW5J3H98FJbAhL5LgADLHUTr31Tm6hdu7C7An0oVOTGrYV/eWAuzlDaV4njFDKP44PKHmONOQv/U86qO6EjLYCL1hJl+NXMYyQYMR5A1WS2er29GnRSX621+/LJxq3VyxWbBcWthr4d464IbTOuB8YDvDF/8HUEsDBBQAAAAIAAAAISik1XLsUgAAAFUAAAAXAAAAc2NyaXB0cy9mZGkvX19pbml0X18ucHkNyzsKgDAMANDdU4Ts7SlEcHHxBDVNIdgm0g/o7e32loeIp41K7K5AN0fY1h1aH/HzcBgk0aAkIbtikTOIJq6sxCANpDxW+zxWgV+mMe0RcfkBUEsDBBQAAAAIAAAAISh8WKhpdQAAAJ8AAAAdAAAAcmVzZWFyY2gvZmRpL3JlcXVpcmVtZW50cy50eHQljkkShSAMBfccJsWkOw6DwsIqhgihvtz+B1120i95ZWSczmnQsAv0Jfj+kRH9vNZKgdoWkaeea4iJBQnKwiayJ0yV0nU4Z0BJ1irGgvNJ74CVJ7WwDnJC4PSt1R+jZJaixXvETu9Do8GycNZRqK1CFnaO/AFQSwECFAAUAAAACAAAACEooInlnFcUAABfNAAAGQAAAAAAAAAAAAAAgAEAAAAAc2NyaXB0cy9mZGlfcm9idXN0bmVzcy5weVBLAQIUABQAAAAIAAAAISgYFrVzjxEAAFctAAAVAAAAAAAAAAAAAACAAY4UAABzY3JpcHRzL2ZkaS9tb2RlbHMucHlQSwECFAAUAAAACAAAACEoE98eYvwHAACNFwAAFgAAAAAAAAAAAAAAgAFQJgAAc2NyaXB0cy9mZGkvc3RvcmFnZS5weVBLAQIUABQAAAAIAAAAISik1XLsUgAAAFUAAAAXAAAAAAAAAAAAAACAAYAuAABzY3JpcHRzL2ZkaS9fX2luaXRfXy5weVBLAQIUABQAAAAIAAAAISh8WKhpdQAAAJ8AAAAdAAAAAAAAAAAAAACAAQcvAAByZXNlYXJjaC9mZGkvcmVxdWlyZW1lbnRzLnR4dFBLBQYAAAAABQAFAF4BAAC3LwAAAAA=')
assert hashlib.sha256(raw).hexdigest()=='2c8c63eb7cc6fb7c4229eacda9540095bb44dff31483c62b36a1da3e5d9098b7'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert z.testzip() is None
    for name in z.namelist():
        assert (PROJECT/name).resolve().is_relative_to(PROJECT.resolve())
    z.extractall(PROJECT)
print('Source SHA256 verified:','2c8c63eb7cc6fb7c4229eacda9540095bb44dff31483c62b36a1da3e5d9098b7')

## Environment (CPU only)
Install the tested pinned baseline dependencies. This notebook uses Python only. Installation does not run any estimator.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--quiet','-r',str(PROJECT/'research/fdi/requirements.txt')],check=True)
import importlib.metadata
VERSIONS={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','statsmodels']}
print(VERSIONS)

## Resume inspection and input authentication
Use the existing study folder. Mounting Drive uses the previously authorized project scope.
Find the exact saved quarterly panel and native national output by their known hashes.
Stop if absent or ambiguous; restore a verified archive rather than reacquiring data or re-estimating.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
STUDY=Path('/content/drive/MyDrive/Brazil FDI Observatory')
EXPECTED={'prepared/quarterly.csv':'10426ad1fef8fcec3a53ff94c1753d289de4faf908437b1ffc787ac3b0d56b8a','models/national.json':'eee06465aa0c4ee366e2476d1083902a4f3a3d1a7713d843e2ad239666a0cc82'}
candidates=[]
for panel in STUDY.rglob('quarterly.csv'):
    run=panel.parent.parent
    if all((run/name).is_file() and hashlib.sha256((run/name).read_bytes()).hexdigest()==v for name,v in EXPECTED.items()):candidates.append(run)
assert candidates,'Matching baseline absent: restore a verified archive. Never rerun it here.'
# Identical verified copies are interchangeable; choose the shortest canonical path and disclose it.
RUN=sorted(candidates,key=lambda p:(len(str(p)),str(p)))[0]
OUTPUT=STUDY/'robustness-extension'
sys.path.insert(0,str(PROJECT/'scripts'))
from fdi_robustness import plan,backup
ROOT=plan(RUN,OUTPUT)
(ROOT/'environment.json').write_text(json.dumps(VERSIONS,indent=2))
print('Verified baseline:',RUN,'Matching copies:',len(candidates),'New identity:',ROOT.name)
for p in sorted(ROOT.glob('*-status.json')):print(p.name,p.read_text())
print('Prepared only; baseline untouched.')

## Execution switches
Keep False for preparation/review. Change to True only when deliberately running this new extension.
Each completed stage is content-hash verified before reuse. Changed code/configuration/input creates a new identity.
The simulation stage is intentionally separate and may take longer; do not use Run All blindly.

In [ ]:
RUN_NEW_CHECKS=False
RUN_SIMULATIONS=False

## Stage: residuals
Outputs are saved in the separate identity. Backups are ZIP-CRC and per-member SHA256 read-back verified.
After this stage, download its backup and independently verify the local copy before proceeding.
No result from this cell is present until the switch is deliberately enabled and execution completes.

In [ ]:
if RUN_NEW_CHECKS:
    subprocess.run([sys.executable,str(PROJECT/'scripts/fdi_robustness.py'),'--run',str(RUN),'--output',str(OUTPUT),'--stage','residuals'],check=True)
else:
    print('Prepared, not executed:','residuals')

In [ ]:
# Separate download action: never combine cell execution with a blocking browser download wait.
from google.colab import files
if ROOT.exists() and (ROOT/'robustness-backup.zip').exists():
    CHECK=json.loads((ROOT/'backup-verification.json').read_text())
    assert hashlib.sha256((ROOT/'robustness-backup.zip').read_bytes()).hexdigest()==CHECK['sha256']
    print(CHECK)
    # Deliberately uncomment only after a completed stage:
    # files.download(str(ROOT/'robustness-backup.zip'))

## Stage: lp
Outputs are saved in the separate identity. Backups are ZIP-CRC and per-member SHA256 read-back verified.
After this stage, download its backup and independently verify the local copy before proceeding.
No result from this cell is present until the switch is deliberately enabled and execution completes.

In [ ]:
if RUN_NEW_CHECKS:
    subprocess.run([sys.executable,str(PROJECT/'scripts/fdi_robustness.py'),'--run',str(RUN),'--output',str(OUTPUT),'--stage','lp'],check=True)
else:
    print('Prepared, not executed:','lp')

In [ ]:
# Separate download action: never combine cell execution with a blocking browser download wait.
from google.colab import files
if ROOT.exists() and (ROOT/'robustness-backup.zip').exists():
    CHECK=json.loads((ROOT/'backup-verification.json').read_text())
    assert hashlib.sha256((ROOT/'robustness-backup.zip').read_bytes()).hexdigest()==CHECK['sha256']
    print(CHECK)
    # Deliberately uncomment only after a completed stage:
    # files.download(str(ROOT/'robustness-backup.zip'))

## Stage: saved-draws
Outputs are saved in the separate identity. Backups are ZIP-CRC and per-member SHA256 read-back verified.
After this stage, download its backup and independently verify the local copy before proceeding.
No result from this cell is present until the switch is deliberately enabled and execution completes.

In [ ]:
if RUN_NEW_CHECKS:
    subprocess.run([sys.executable,str(PROJECT/'scripts/fdi_robustness.py'),'--run',str(RUN),'--output',str(OUTPUT),'--stage','saved-draws'],check=True)
else:
    print('Prepared, not executed:','saved-draws')

In [ ]:
# Separate download action: never combine cell execution with a blocking browser download wait.
from google.colab import files
if ROOT.exists() and (ROOT/'robustness-backup.zip').exists():
    CHECK=json.loads((ROOT/'backup-verification.json').read_text())
    assert hashlib.sha256((ROOT/'robustness-backup.zip').read_bytes()).hexdigest()==CHECK['sha256']
    print(CHECK)
    # Deliberately uncomment only after a completed stage:
    # files.download(str(ROOT/'robustness-backup.zip'))

## Stage: simulation
Outputs are saved in the separate identity. Backups are ZIP-CRC and per-member SHA256 read-back verified.
After this stage, download its backup and independently verify the local copy before proceeding.
No result from this cell is present until the switch is deliberately enabled and execution completes.

In [ ]:
if RUN_SIMULATIONS:
    subprocess.run([sys.executable,str(PROJECT/'scripts/fdi_robustness.py'),'--run',str(RUN),'--output',str(OUTPUT),'--stage','simulation'],check=True)
else:
    print('Prepared, not executed:','simulation')

In [ ]:
# Separate download action: never combine cell execution with a blocking browser download wait.
from google.colab import files
if ROOT.exists() and (ROOT/'robustness-backup.zip').exists():
    CHECK=json.loads((ROOT/'backup-verification.json').read_text())
    assert hashlib.sha256((ROOT/'robustness-backup.zip').read_bytes()).hexdigest()==CHECK['sha256']
    print(CHECK)
    # Deliberately uncomment only after a completed stage:
    # files.download(str(ROOT/'robustness-backup.zip'))

## Release gate
No automatic publication exists. A completed output is not automatically scientifically validated.
Inspect all failures, estimates, interval definitions, assumptions, source hashes and simulated coverage uncertainty.
A new national check cannot cure missing municipal exposure histories. Retain null/inconclusive results.
Upload only validated aggregate exports to the website; keep books, private microdata and research archives private.